# ADoRE-DoE Phase 3：Active Learning（Module 1）

GPR 代理 → 預測 R² → 多響應停止條件 → 建議下一批實驗點 → 更新。
> 模擬結果由**合成函數（Synthetic）**產生，三個響應的單位相差數個數量級；實際使用時請換成你自己的模擬結果。安裝：`pip install -e ".[plot]"`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import adore_doe as ad
from adore_doe import design, active_learning as al
SEED = 2026

## 1. 因子、合成「模擬器」、初始設計

In [ ]:
factors = [ad.Factor("Width", 10, 50, unit="nm"), ad.Factor("Spacing", 5, 30, unit="nm"), ad.Factor("Thickness", 20, 80, unit="nm")]

def simulate(Z):                       # Z: coded points -> three responses with very different scales (Synthetic)
    z = np.atleast_2d(Z)
    return {"Capacitance": 2e-15 * (1 + z[:, 0] - 0.5 * z[:, 1] + 0.4 * z[:, 0] * z[:, 1] + 0.6 * z[:, 2] ** 2),
            "Resistance":  120 + 30 * np.sin(2 * z[:, 0]) + 25 * z[:, 1] ** 2 - 10 * z[:, 2],
            "Delay":       1e-9 * (1 + 0.3 * z[:, 0] + 0.2 * np.cos(3 * z[:, 1]) * z[:, 2])}

ds = design.i_optimal(factors, "quadratic", runs=12, seed=SEED)
for name, v in simulate(ds.coded).items():
    ds.add_response(name, v)
print(ds.n_runs, "initial runs; responses:", ds.response_names)

## 2. 停止條件與 Active Learner

每個響應的預測 R² 都要 ≥ 0.90，聚合（平均）R² 也要 ≥ 0.93，且**連續 2 輪**；最多 30 個 run、每輪 2 個。`budget_exhausted` 代表**目標未達成**。

In [ ]:
stop = al.MultiResponseStopping(n_max=30, response_min_r2=0.90, aggregate_r2=0.93, consecutive=2, n_min=12, batch_size=2)
evalset = al.EvaluationSet.build(factors, 800, seed=99).with_truth(simulate)     # 固定、獨立的評估集（此處有真值，因為是合成函數）
L = al.ActiveLearner(ds, ["Capacitance", "Resistance", "Delay"], stop, strategy="ivr", seed=SEED,
                     n_candidates=400, n_restarts=2, eval_set=evalset)

## 3. 手動走一輪：assess → recommend（只建議、不執行模擬）

In [ ]:
d = L.assess()
print(d.reason, "|", d.message)
sel = L.recommend()
print("這一步沒有執行任何模擬；資料集仍有", L.ds.n_runs, "個 run")
sel.actual_df(factors).round(2)

In [ ]:
L.update(sel.points, simulate(sel.points))                # 使用者執行模擬後，把結果交回去
print(L.ds.n_runs, "runs; 新增 run 的來源:", L.ds.source[-1])

## 4. 自動跑到停止（以合成函數代替真實模擬）

In [ ]:
d = L.run(simulate)
print(d.reason, "| target_met =", d.target_met)
print(d.message)
L.summary()

In [ ]:
L.history_df()[["round", "n_runs", "r2_Capacitance", "r2_Resistance", "r2_Delay", "aggregate_r2", "decision"]].round(3)

負的 R² 會保留（早期輪次樣本很少）；`aggregate_r2` 為 None 代表該輪有響應的 R² 未定義或尚未計算。

## 5. 學習曲線

In [ ]:
h = L.history_df()
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for r in ["Capacitance", "Resistance", "Delay"]:
    ax[0].plot(h.n_runs, h[f"r2_{r}"], marker="o", label=r)
    ax[1].plot(h.n_runs, h[f"eval_rmse_{r}"] / evalset.truth[r].std(), marker="o", label=r)
ax[0].axhline(0.90, color="gray", ls="--", label="threshold 0.90"); ax[0].set_ylim(-1, 1.05)
ax[0].set_xlabel("runs"); ax[0].set_ylabel("cross-validated predictive R²"); ax[0].legend(fontsize=8)
ax[1].set_xlabel("runs"); ax[1].set_ylabel("true NRMSE on the evaluation set"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig("output/active_learning_curves.png", dpi=150); plt.show()

左圖是**停止條件使用的**交叉驗證預測 R²（實驗現場能得到的）；右圖是用**真值**算的評估集誤差（只有合成函數或已有大量資料時才有）。

## 6. 可重現性與預算統計

In [ ]:
print("root seed:", L.sm.root, "| runs:", L.n_runs, "| unique points:", L.ds.training().n_unique_points,
      "| replicated runs:", L.ds.training().n_replicated_runs)
print("sources:", sorted(set(L.ds.source)))